## **Install Required Inference Dependencies**

Install `demucs` along with `transformers`, `torchaudio`, `soundfile`, and `librosa` to support speech separation and audio processing for inference.

In [1]:
!pip install -q demucs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.1/249.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 4.8 MB/s eta 0:00:00


In [2]:
!pip -q install transformers torchaudio soundfile librosa

### **Necessary imports**

In [3]:
import os
import glob
import shutil
import pathlib
import subprocess
import torch
import torchaudio
import librosa
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

2026-02-22 07:01:35.032548: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771743695.263558      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771743695.326110      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771743695.871902      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771743695.871951      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771743695.871955      25 computation_placer.cc:177] computation placer alr

## **Set Up Test Paths and Working Directories**

This cell defines the test audio directory, creates necessary working folders for processing and outputs, and collects the list of test `.wav` files for transcription.

In [4]:
# Test directory
test_audio_dir = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"

# Working folders
work_audio_dir = "/kaggle/working/audio_full"
demucs_out_dir = "/kaggle/working/separated_full"
out_txt_dir = "/kaggle/working/test_transcripts"

os.makedirs(work_audio_dir, exist_ok=True)
os.makedirs(demucs_out_dir, exist_ok=True)
os.makedirs(out_txt_dir, exist_ok=True)

wav_files = sorted(glob.glob(os.path.join(test_audio_dir, "*.wav")))
print("Number of test files:", len(wav_files))
print("Example:", wav_files[0] if wav_files else "None")

Number of test files: 24
Example: /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio/test_001.wav


## **Load Silero Voice Activity Detection Model**

This cell loads the Silero VAD model via Torch Hub and retrieves the speech timestamp utility function for detecting speech segments in audio files.

In [5]:
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)

(get_speech_timestamps, _, _, _, _) = utils
print("Silero VAD loaded")

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
Silero VAD loaded


## **Load Local Whisper Model for ASR**

This cell locates the Whisper model files, loads the processor and model from the detected path, moves the model to the appropriate device, and prepares it for transcription.

In [6]:
base = "/kaggle/input/datasets/tugstugi/bengali-ai-asr-submission/bengali-whisper-medium"

# Auto-detect HF model folder
candidates = []
for root, _, files in os.walk(base):
    if "config.json" in files and ("pytorch_model.bin" in files or "model.safetensors" in files):
        candidates.append(root)

if not candidates:
    raise FileNotFoundError("Whisper model not found")

model_path = candidates[0]
print("Using model:", model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
model.eval()

Using model: /kaggle/input/datasets/tugstugi/bengali-ai-asr-submission/bengali-whisper-medium
Device: cuda


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 1024, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(1024, 1024, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 1024)
      (layers): ModuleList(
        (0-23): 24 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=Tru

## **Utilities for VAD-Based Chunking**

This cell defines helper functions to group VAD speech segments into ~20s windows without cutting mid-speech, and to extract audio chunks with optional padding for more stable ASR decoding.

In [7]:
def build_vad_windows(segments, target_len_s=20.0, max_gap_s=5.0):
    """
    Builds timeline windows:
    - Target ~20s
    - Never cut inside a segment
    - Do NOT merge if silence gap > max_gap_s
    """
    windows = []
    i, n = 0, len(segments)

    while i < n:
        win_start = segments[i][0]
        win_end = segments[i][1]
        i += 1

        while i < n:
            next_start, next_end = segments[i]
            gap = next_start - win_end

            # Stop if silence too large
            if gap > max_gap_s:
                break

            win_end = next_end
            i += 1

            if (win_end - win_start) >= target_len_s:
                break

        windows.append((win_start, win_end))

    return windows


def slice_with_padding(wav_1ch, sr, start_s, end_s, pad_s=2.0):
    start_i = max(0, int(round(start_s * sr)))
    end_i = min(wav_1ch.shape[-1], int(round(end_s * sr)))
    chunk = wav_1ch[:, start_i:end_i]

    pad_len = int(round(pad_s * sr))
    pad = torch.zeros((1, pad_len), dtype=chunk.dtype)

    return torch.cat([pad, chunk, pad], dim=1)

## **Demucs Vocal Separation Function**

This cell defines a function that runs Demucs (`htdemucs`) to extract the vocal stem from an input audio file and returns the path to the separated vocals for downstream processing.

In [8]:
TARGET_SR = 16000

def demucs_two_stem_vocals(in_wav_path):
    cmd = [
        "python", "-m", "demucs",
        "-n", "htdemucs",
        "--two-stems", "vocals",
        "-o", demucs_out_dir,
        in_wav_path
    ]
    subprocess.run(cmd, check=True)

    track_name = pathlib.Path(in_wav_path).stem
    vocals_path = f"{demucs_out_dir}/htdemucs/{track_name}/vocals.wav"
    return vocals_path

## **Run Full ASR Pipeline and Generate Transcripts**

This cell processes each test audio file by performing Demucs vocal separation, applying VAD-based chunking, transcribing segments with Whisper, saving individual transcript files, and compiling the final submission dataframe.

In [9]:
all_results = {}
cnt = 0
for in_wav in tqdm(wav_files, desc="Processing Files"):
    fname = pathlib.Path(in_wav).name
    stem = pathlib.Path(in_wav).stem


    # Demucs 2-stem
    vocals_path = demucs_two_stem_vocals(in_wav)

    # Load vocals
    wav, sr = torchaudio.load(vocals_path)
    wav = wav.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    sr = TARGET_SR

    audio_1d = wav.squeeze(0).cpu()

    # VAD
    speech_ts = get_speech_timestamps(audio_1d, vad_model, sampling_rate=sr)
    segments = [(s["start"]/sr, s["end"]/sr) for s in speech_ts]

    if not segments:
        transcript = ""
    else:
        # Build windows with gap protection
        windows = build_vad_windows(
            segments,
            target_len_s=20.0,
            max_gap_s=5.0
        )

        parts = []

        for ws, we in tqdm(windows, desc=f"Windows {fname}", leave=False):
            chunk = slice_with_padding(wav, sr, ws, we, pad_s=1.0)
            audio_np = chunk.squeeze(0).cpu().numpy()

            inputs = processor(audio_np, sampling_rate=sr, return_tensors="pt")
            feats = inputs.input_features.to(device)

            with torch.no_grad():
                pred_ids = model.generate(
                    feats,
                    max_new_tokens=256,
                )

            text = processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()
            if text:
                parts.append(text)

        transcript = " ".join(parts)  

    # Save individual txt
    out_path = os.path.join(out_txt_dir, f"{stem}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(transcript)

    # Store for submission
    all_results[stem] = transcript

    # Cleanup Demucs output
    track_folder = os.path.join(demucs_out_dir, "htdemucs", stem)
    if os.path.exists(track_folder):
        shutil.rmtree(track_folder)
    cnt = cnt+1
    print(f"{cnt} files transcribed")

print("Done. Individual transcripts saved.")


# ----------------------------------
# Create submission CSV
# ----------------------------------
submission_df = pd.DataFrame({
    "filename": list(all_results.keys()),
    "transcript": list(all_results.values())
})

submission_df = submission_df.sort_values("filename").reset_index(drop=True)

Processing Files:   0%|          | 0/24 [00:00<?, ?it/s]

Downloading: "https://dl.fbaipublicfiles.com/demucs/hybrid_transformer/955717e8-8726e21a.th" to /root/.cache/torch/hub/checkpoints/955717e8-8726e21a.th


100%|██████████| 80.2M/80.2M [00:00<00:00, 142MB/s]


Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /kaggle/working/separated_full/htdemucs
Separating track /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio/test_001.wav


100%|████████████████████████████████████████████████████████████████████| 3375.45/3375.45 [02:13<00:00, 25.21seconds/s]
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Ple

...

Windows test_030.wav:   0%|          | 0/144 [00:00<?, ?it/s]

23 files transcribed
Selected model is a bag of 1 models. You will see that many progress bars per track.
Separated tracks will be stored in /kaggle/working/separated_full/htdemucs
Separating track /kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio/test_032.wav


100%|██████████████████████████████████████████████| 2597.3999999999996/2597.3999999999996 [01:51<00:00, 23.22seconds/s]
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:337: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.save_with_torchcodec` under the hood. Some parameters like format, encoding, bits_per_sample, buffer_size, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's encoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.encoders.AudioEncoder
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:247: UserWarning: torio.io._streaming_media_encoder.StreamingMediaEncoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Ple

Windows test_032.wav:   0%|          | 0/116 [00:00<?, ?it/s]

24 files transcribed
Done. Individual transcripts saved.


## **Normalize Transcripts and Save Submission**

This cell applies Unicode normalization and whitespace cleanup to Bengali transcripts, removes hidden zero-width characters, and saves the final sorted submission file as `submission.csv`.

In [10]:
import re
import unicodedata

ZW = r"[\u200B-\u200D\uFEFF]"  # zero-width chars + BOM

def normalize_bn_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)   # Unicode normalize
    s = re.sub(ZW, "", s)                 # remove zero-width chars
    s = s.replace("\u00A0", " ")          # replace non-breaking space
    s = " ".join(s.split())               # collapse extra whitespace
    return s

submission_df["transcript"] = submission_df["transcript"].apply(normalize_bn_text)

submission_df = submission_df.sort_values("filename").reset_index(drop=True)

submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False, encoding="utf-8")

print("Submission saved to:", submission_path)

Submission saved to: submission.csv
